# QSAR — predicting COX-2 potency from structure
### Train a model on real ChEMBL data, then score your ferulic library

**The idea (QSAR = Quantitative Structure-Activity Relationship):** if you collect many molecules whose COX-2 inhibition has been *measured*, a model can learn the structure -> activity mapping and then predict potency for *new* molecules — including FA19 — **without docking or synthesising them.** Docking asks "does it fit the pocket?"; QSAR asks "what does the data say molecules like this do?" They are complementary evidence.

**Plan:** pull COX-2 IC50 data from ChEMBL -> clean it -> convert to pIC50 -> turn molecules into fingerprints -> train + honestly validate a model -> predict your library.

> Runs anywhere with internet (ChEMBL is a web API). Colab is easiest.


## 1 · Install
*Why:* the ChEMBL web-service client fetches the data; RDKit makes fingerprints; scikit-learn is the model + metrics.

In [ ]:
!pip -q install chembl_webresource_client rdkit scikit-learn pandas numpy matplotlib 2>/dev/null
print("installed")

## 2 · Find the COX-2 target
Every target in ChEMBL has an ID. Human COX-2 (gene *PTGS2*) is **CHEMBL230**. We search by name to confirm it rather than trusting a hard-coded ID — defensive and transparent.


In [ ]:
from chembl_webresource_client.new_client import new_client
target = new_client.target
hits = target.search("cyclooxygenase-2")
import pandas as pd
tdf = pd.DataFrame(hits)[["target_chembl_id","pref_name","organism","target_type"]]
print(tdf.head(8).to_string(index=False))
TARGET_ID = "CHEMBL230"   # human COX-2 / PTGS2 (confirm it appears above)

## 3 · Pull the bioactivities
We request **IC50** values on this target. *Why we filter hard here:*
- **`standard_type = IC50`** — mixing IC50, Ki, EC50 measures different things; keep one for a clean target variable.
- **`standard_relation = '='`** — drop "greater than"/"less than" censored values; they aren't exact numbers a regressor can learn from.
- **`standard_units = nM`** — one unit so values are comparable.

Each kept row is: a molecule (SMILES) + a measured IC50.


In [ ]:
activity = new_client.activity
acts = activity.filter(target_chembl_id=TARGET_ID, standard_type="IC50").only(
    ["molecule_chembl_id","canonical_smiles","standard_value","standard_units","standard_relation"])
raw = pd.DataFrame(acts)
print("raw activity rows:", len(raw))
raw.head()

## 4 · Clean and convert to pIC50
*Why pIC50:* potency spans many orders of magnitude (1 nM to 100,000 nM). On a raw scale a model is dominated by the huge numbers. **pIC50 = -log10(IC50 in molar)** compresses that into a tidy, roughly normal range (~4-10) and is linear in binding free energy — the standard QSAR target.

For IC50 in nM:  **pIC50 = 9 - log10(IC50_nM)**.  We also deduplicate molecules (a compound measured several times) by taking the median.


In [ ]:
import numpy as np
df = raw.dropna(subset=["canonical_smiles","standard_value","standard_units"]).copy()
df = df[(df["standard_units"]=="nM") & (df["standard_relation"]=="=")]
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df[(df["standard_value"]>0) & (df["standard_value"]<1e7)]          # drop 0 / absurd
df["pIC50"] = 9 - np.log10(df["standard_value"])                        # nM -> pIC50
# one value per molecule (median of replicates)
df = df.groupby("canonical_smiles", as_index=False)["pIC50"].median()
print(f"clean, unique molecules: {len(df)}")
print(df["pIC50"].describe().round(2))

## 5 · Featurise — molecules into numbers
A model needs vectors, not SMILES. We use **Morgan / ECFP4 fingerprints** (radius 2, 2048 bits): each bit flags the presence of a particular local substructure. *Why these:* they capture the chemical fragments that drive activity and are the proven workhorse for similarity-based property models.

*(We use the modern `MorganGenerator` API — the older `GetMorganFingerprintAsBitVect` works too but is deprecated.)*


In [ ]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def featurise(smiles_list):
    X, keep = [], []
    for i, smi in enumerate(smiles_list):
        m = Chem.MolFromSmiles(str(smi))
        if m is None: continue
        arr = np.zeros((2048,), dtype=int)
        DataStructs.ConvertToNumpyArray(gen.GetFingerprint(m), arr)
        X.append(arr); keep.append(i)
    return np.array(X), keep

X, keep = featurise(df["canonical_smiles"])
y = df["pIC50"].values[keep]
print("feature matrix:", X.shape, "| labels:", y.shape)

## 6 · Train and honestly validate
**Model: Random Forest.** *Why:* it handles high-dimensional sparse fingerprints well, needs little tuning, resists overfitting, and is a strong baseline. (Upgrades: XGBoost; or a graph neural network like Chemprop for the state of the art.)

**Validation: 5-fold cross-validation.** *Why not a single split:* one lucky/unlucky split gives a misleading score; CV averages over five.

> **Caveat that matters:** a *random* split lets near-identical analogues land in both train and test, leaking information and **inflating** the score. The honest version is a **scaffold split** (group by core skeleton). We show the random-split number for simplicity but flag this — report scaffold-split performance in anything serious.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

model = RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=-1)
cv = KFold(n_splits=5, shuffle=True, random_state=0)
y_cv = cross_val_predict(model, X, y, cv=cv)
print(f"CV R2   = {r2_score(y, y_cv):.3f}")
print(f"CV RMSE = {mean_squared_error(y, y_cv)**0.5:.3f} pIC50 units")

plt.figure(figsize=(5.5,5.5))
plt.scatter(y, y_cv, s=10, alpha=0.4, color="#2a9d8f")
lim=[min(y.min(),y_cv.min()), max(y.max(),y_cv.max())]
plt.plot(lim, lim, "k--", lw=1)
plt.xlabel("measured pIC50"); plt.ylabel("cross-validated prediction")
plt.title("QSAR model — predicted vs measured"); plt.tight_layout(); plt.show()

model.fit(X, y)   # refit on all data for downstream prediction

## 7 · Applicability domain — when to *not* trust a prediction
A model is reliable only for molecules resembling its training set. We measure that with **maximum Tanimoto similarity** of each new molecule to any training compound. A rule of thumb: similarity below ~0.3 means "out of domain — treat the prediction with suspicion."


In [ ]:
train_fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in df["canonical_smiles"].values[keep]]
def max_sim(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return np.nan
    sims = DataStructs.BulkTanimotoSimilarity(gen.GetFingerprint(m), train_fps)
    return max(sims)

## 8 · Predict your library
Now the payoff: predict COX-2 pIC50 for FA19 and the derivatives, attach the applicability-domain flag, and rank. Higher pIC50 = more potent predicted inhibitor.


In [ ]:
try:
    lib = pd.read_csv("data/library.csv")
except FileNotFoundError:
    lib = pd.DataFrame([{"id":"FA19","name":"FA19 lead",
        "smiles":"COc1ccc(C(=O)Oc2ccc(/C=C/C(=O)O)cc2OC)cc1"}])
Xl, keepl = featurise(lib["smiles"])
lib = lib.iloc[keepl].copy()
lib["pred_pIC50"]   = model.predict(Xl).round(2)
lib["max_train_sim"]= [round(max_sim(s),2) for s in lib["smiles"]]
lib["in_domain"]    = lib["max_train_sim"] >= 0.30
lib = lib.sort_values("pred_pIC50", ascending=False)
lib.to_csv("data/qsar_predictions.csv", index=False)
print(lib[["id","name","pred_pIC50","max_train_sim","in_domain"]].to_string(index=False))

## 9 · Honest limits & how it fits
- **The score is the ceiling of trust.** If CV R2 is low (say < 0.4) or your molecules are out of domain, the predictions are weak evidence — say so.
- **Random-split metrics flatter the model;** report scaffold-split for a real claim.
- **QSAR complements docking, it doesn't replace it.** Two independent lines (a data-driven potency estimate + a physics-based binding pose) agreeing is a much stronger story than either alone.
- **Portfolio framing:** "I built a COX-2 QSAR model on N ChEMBL compounds (CV R2 = ...), confirmed FA19 sits inside its applicability domain, and used it to rank the library — consistent with the docking selection of FA19."
